# Data Cleaning ideas and testing 

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

c:\Users\ldcal\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1. importing the datasets 

In [27]:
activity_data = pd.read_csv(r"Data\activity_sample.csv", low_memory=False)
contact_data = pd.read_csv(r"Data\contact_sample.csv", low_memory=False)
sdk_download_data = pd.read_csv(r"Data\sdk_download_sample.csv", low_memory=False)

## 2. Finding columns with missing data 

In [29]:
# finding columns with missing data
print(activity_data.columns[activity_data.isnull().any()])

Index(['activity_name', 'activity_role', 'activity_attendance',
       'activity_score', 'filepath', 'pk2', 'lead_source',
       'nvidia_campaign_id', 'gtc_nvidia_campaign_id', 'lead_source_details'],
      dtype='str')


In [30]:
print(activity_data["activity_attendance"].value_counts(dropna=False))

activity_attendance
NaN                                            81205
true                                           11096
Attended Live                                   2791
Attended Live;Registered                        2453
Attended On-Demand;Registered                   1949
Registered;Attended On-Demand                    191
Registered; Attended Live                        181
Registered;Attended Live                          71
Attended Live;Attended On-Demand;Registered       27
Attended On-Demand                                19
Registered;Attended Live;Attended On-Demand        7
Attended Live;Attended On-Demand                   3
Attended On-Demand;Attended Live;Registered        3
Attended On-Demand;Registered;Attended Live        3
Attended Live;Registered;Attended On-Demand        1
Name: count, dtype: int64


In [19]:
activity_data["activity_attendance"].value_counts(dropna=False).sum()

np.int64(100000)

## 3. Adding a column for registered

In [33]:
activity_data["registered"] = activity_data["activity_attendance"].str.contains("Registered", case=False, na=False)

activity_data["registered"].value_counts()

registered
False    95114
True      4886
Name: count, dtype: int64

This can be a uniqye feature that may give us more insights into if the user is being recorded properly into the activity dataset and can be used to help segment developers

## 4. Cleaning `activity_attendance` - condensed columns 

In [14]:
def classify_attendance(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).lower()
    if "attended live" in s and "attended on-demand" in s:
        return "Attended Live & On-Demand"
    if "attended live" in s:
        return "Attended Live"
    if "attended on-demand" in s:
        return "Attended On-Demand"
    if "true" in s:
        return "True"
    return "Other"

activity_data["activity_attendance"] = activity_data["activity_attendance"].apply(classify_attendance)
print(activity_data["activity_attendance"].value_counts(dropna=False))

activity_attendance
NaN                          81205
True                         11096
Attended Live                 5496
Attended On-Demand            2159
Attended Live & On-Demand       44
Name: count, dtype: int64


the idea behind this cleaning was to combine a lot of the rows that contained simmilar semantics, whilst still being able to show the rows that multiple things, such as live and on demand, condensing these rows can help us overall. 

In [20]:
activity_data["activity_attendance"].value_counts(dropna=False).sum()

np.int64(100000)

In [21]:
activity_data.head(20)

,dev_contact,activity,activity_name,activity_type,activity_role,activity_attendance,activity_score,activity_date,activity_id,filepath,pk1,pk2,lead_source,nvidia_campaign_id,gtc_nvidia_campaign_id,lead_source_details
0,6482037,DevZone Downloads,JetPack SDK,Installer,DZ4,NaN,3.0,2025-11-07 00:00:00,DZ4:2806383_22289_20251107,/assets/embedded/secure/tools/files/jetpack-sd...,DZ4:2806383_22289_20251107,22289,NaN,NaN,NaN,NaN
1,10132282,Model API,ai-llama-3_3-70b-instruct,API Catalog Hosted API,NVCF,NaN,0.0,2026-02-04 00:00:00,48OOZMrWCGufvT4vw8fNGcVd0ZNONGaj8FjzYLHBYCI_20...,84eb5de1-166b-4bb4-a01b-4f51bd90aa52,48OOZMrWCGufvT4vw8fNGcVd0ZNONGaj8FjzYLHBYCI_20...,84eb5de1-166b-4bb4-a01b-4f51bd90aa52,NaN,NaN,NaN,NaN
2,4624355,DevZone Downloads,jetpack,Installer,Devzone,NaN,3.0,2022-04-30 00:00:00,4578131_906805_20220430,verizon://assets/embedded/secure/tools/files/j...,4578131_906805_20220430,906805,NaN,NaN,NaN,NaN
3,10573410,Dev Program Membership,Developer,Approved,NaN,true,1.0,2026-03-12 05:08:35,874688_17198669,NaN,874688_17198669,874688,NaN,NaN,NaN,NaN
4,4653885,User Feedback,cuDNN Download Survey,Feedback Survey,NaN,NaN,3.0,2022-05-09 00:00:00,SURVEY_894471-4653885,NaN,SURVEY_894471-4653885,SURVEY_894471,website,NaN,NaN,NaN
5,6740502,DevZone Downloads,JetPack SDK,Installer,DZ4,NaN,3.0,2024-05-28 00:00:00,DZ4:4794959_18957_20240528,/assets/embedded/secure/tools/files/jetpack-sd...,DZ4:4794959_18957_20240528,18957,NaN,NaN,NaN,NaN
6,3188041,Dev Program Membership,Developer,Approved,NaN,true,1.0,2023-01-19 09:35:20,874688_6837130,NaN,874688_6837130,874688,NaN,NaN,NaN,NaN
7,8922078,DevZone Downloads,JetPack SDK,Installer,DZ4,NaN,3.0,2025-05-27 00:00:00,DZ4:6418878_26376_20250527,/assets/embedded/secure/tools/files/jetpack-sd...,DZ4:6418878_26376_20250527,26376,NaN,NaN,NaN,NaN
8,453036,Dev Program Membership,Developer,Approved,NaN,true,1.0,2020-06-14 08:24:36,874688_2601035,NaN,874688_2601035,874688,NaN,NaN,NaN,NaN
9,1536056,DevZone Downloads,cuDNN,Installer,Devzone,NaN,3.0,2021-01-09 00:00:00,2220048_886734_20210109,verizon://compute/machine-learning/cudnn/secur...,2220048_886734_20210109,886734,NaN,NaN,NaN,NaN


## 5. Condensing `activity_role` column

In [22]:
activity_data["activity_role"].value_counts()

activity_role
DZ4                  29928
Devzone              27369
NGC                  11087
NVCF                  9224
Attendee              7523
User                   740
Moderator              422
Other                   63
P0-Must have            55
Exhibitor               49
Unprioritized           45
Presenter/Speaker       38
P1-Should have          37
Administrator           24
5-Normal                18
Speaker                 11
P2-Nice to have          6
Financial Analyst        3
Press                    3
4-Fix Before Ship        1
Student                  1
*Showstopper*            1
Name: count, dtype: int64

In [26]:
activity_data["activity_role"] = activity_data["activity_role"].where(
    activity_data["activity_role"].isin(activity_data["activity_role"].value_counts()[lambda x: x >= 70].index), other="Other"
)

activity_data["activity_role"].value_counts()

activity_role
DZ4          29928
Devzone      27369
Other        13707
NGC          11087
NVCF          9224
Attendee      7523
User           740
Moderator      422
Name: count, dtype: int64